In [ ]:
from src.ffnn import *
import pandas as pd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Evaluation
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay
data = pd.read_csv('datasetml_2026.csv')


: 

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
# Cek missing values
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(2)
pd.DataFrame({'Missing Count': missing, 'Missing (%)': missing_pct})

In [ ]:
print(data['placement_status'].value_counts())

sns.countplot(data['placement_status'])

In [ ]:
import math
numerical_cols = data.select_dtypes(include=["int64", "float64"]).columns.tolist()
n_cols = 3 
n_rows = math.ceil(len(numerical_cols) / n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.boxenplot(
        data=data,
        x='placement_status',
        y=col,
        ax=axes[i]
    )
    axes[i].set_title(f"{col} vs placement_status")
    axes[i].set_xlabel('placement_status')
    axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
categorical_cols = data.select_dtypes(include=["object", "category"]).columns.tolist()
categorical_cols.remove('placement_status')
n_cols = 3
n_rows = math.ceil(len(categorical_cols) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    sns.histplot(
        data=data,
        x=col,
        hue='placement_status',
        multiple="dodge",
        shrink=0.8,
        discrete=True,
        ax=axes[i]
    )
    
    axes[i].set_title(f"{col} vs placement_status")
    axes[i].set_xlabel(col)
    axes[i].set_ylabel("Count")
    axes[i].tick_params(axis='x', rotation=45)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, col in zip(axes.flatten(), numerical_cols):
    sns.boxplot(data=data, x='placement_status', y=col, ax=ax)
    ax.set_title(f'{col} by Loan Status')

plt.tight_layout()
plt.show()

In [ ]:
#preprocessing
TARGET_COL = 'placement_status'

print("Categorical:", categorical_cols + ['placement_status'])
print("Numerical:  ", numerical_cols)

# Buat salinan
X = data.drop(columns=[TARGET_COL])
y = data[TARGET_COL]

In [ ]:
label_encoder = LabelEncoder()
y_enc = label_encoder.fit_transform(y)

label_mapping = dict(
    zip(
        label_encoder.classes_,
        label_encoder.transform(label_encoder.classes_),
    )
)
print("Label mapping:", label_mapping)

# Encode semua fitur kategorikal secara cepat
X_enc = pd.get_dummies(X, drop_first=True)

In [ ]:
X_train_df, X_val_df, y_train_np, y_val_np = train_test_split(
    X_enc,
    y_enc,
    test_size=0.2,
    random_state=42,
)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df)
X_val = scaler.transform(X_val_df)

X_train = X_train.astype(np.float32)
X_val = X_val.astype(np.float32)
y_train = y_train_np.astype(np.float32).reshape(-1, 1)
y_val = y_val_np.astype(np.float32).reshape(-1, 1)

print("X_train:", X_train.shape, "| y_train:", y_train.shape)
print("X_val:  ", X_val.shape,   "| y_val:  ", y_val.shape)

In [ ]:
def get_iqr_bounds(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

for col in numerical_cols:
    lower, upper = get_iqr_bounds(data[col])
    n = ((data[col] < lower) | (data[col] > upper)).sum()
    print(f"{col}: {n} outliers")


In [ ]:
input_dim = int(X_train.shape[1])
model = FFNN([input_dim, 12, 6, 3, 1], [Tanh, Tanh, Tanh, Sigmoid])

EPOCHS = 50
model.train(X_train, y_train, EPOCHS)

In [ ]:
results = {}

def evaluate_model(model, X_val, y_val, model_name="Model"):
    model.predict(X_val)
    y_prob = model.result()
    y_pred = (y_prob >= 0.5).astype(int).reshape(-1)
    y_true = y_val.astype(int).reshape(-1)

    print(f"{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(
        classification_report(
            y_true,
            y_pred,
            target_names=["not placed (0)", "placed (1)"],
        )
    )
    macro_f1 = f1_score(y_true, y_pred, average="macro")
    print(f">>> Macro F1-Score: {macro_f1:.4f}\n")
    return macro_f1